In [1]:
from embedder import Embedder

2026-07-10 14:54:50.307139456 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
embed = Embedder()


The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

In [3]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [4]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [5]:
v2.dot(dv)

np.float64(0.019730422395141473)

The first score is higher because the query about joining the course is more similar to the document about registration.

Embed the following query:

How does approximate nearest neighbor search work?

In [6]:
q3 = "How does approximate nearest neighbor search work?"

v3 = embed.encode(q3)

In [7]:
v3

array([-2.05820344e-02, -1.40458849e-02,  3.02994061e-02, -5.40378445e-02,
        7.18781100e-02, -2.79537512e-02, -5.03093823e-02, -1.27217287e-02,
        4.08207902e-02, -2.60037446e-02,  3.05458646e-02,  4.21485309e-02,
        8.09861910e-02, -6.93957355e-02, -1.30190518e-01, -6.39247860e-02,
        4.81059741e-02,  1.60095554e-02, -5.22432468e-02, -7.13635281e-02,
       -3.83859209e-03,  2.48125508e-02,  4.40211692e-02, -3.45579077e-02,
        1.52686257e-02,  7.61533350e-03,  5.38177679e-02,  1.18252557e-02,
        1.32434005e-02,  2.89461963e-02,  6.64912054e-03,  7.04788357e-02,
        6.19290508e-02,  2.11051780e-02, -7.33482889e-02,  2.84129213e-02,
       -3.72108635e-02,  6.22799314e-02, -4.86973136e-02,  4.49663910e-02,
       -2.59481060e-02,  2.04324269e-02,  1.79650458e-02,  1.02705602e-02,
        2.74898095e-03,  2.84324992e-02, -3.30318167e-02,  6.70969402e-02,
       -1.56520555e-02, -8.51907638e-02, -1.24307135e-01,  4.32504103e-02,
       -5.64433014e-02,  

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [8]:
v3[0]

np.float64(-0.02058203437252893)

## Loading the data

We pull the lesson pages from the course repository

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
len(documents)

72

In [11]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. 

In [12]:
# The target value you are looking for
target_val = "02-vector-search/lessons/07-sqlitesearch-vector.md"

# Get the index
index = next((i for i, doc in enumerate(documents) if doc.get("filename") == target_val), None)

print(index)

22


In [13]:
documents[22]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [14]:
index_22_content = documents[22].get("content")

In [15]:
index_22_content_vector = embed.encode(index_22_content)

In [16]:
v3.dot(index_22_content_vector)

np.float64(0.36107026789538205)

# Evaluation

### Generating ground truth
For each lesson page, we ask an LLM to write 5 questions that are answered by that page. Each question is then labeled with the page it came from.

In [4]:
# Generating questions with structured output

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
# Call the LLM for one document

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [20]:
doc = documents[0]
doc

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
# Prepare the document as JSON
import json

doc = documents[0]

user_prompt = json.dumps(doc)

In [8]:
# Create the messages
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [23]:
# For structured output we switch to responses.parse
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [51]:
result = response.output_parsed

print(result)

questions=["What is a retrieval-augmented generation system, and why do people use it instead of relying only on the model's memory?", 'Why does the lesson say to treat an LLM like a black box, and how do you interact with it in this course?', 'What problems with LLMs does RAG help solve, like missing knowledge or hallucinated answers?', "What are the main steps you'll build in the first part of the module to make the FAQ agent work?", 'How is the agentic version in part 2 different from the fixed pipeline built in part 1?']


In [52]:
result

Questions(questions=["What is a retrieval-augmented generation system, and why do people use it instead of relying only on the model's memory?", 'Why does the lesson say to treat an LLM like a black box, and how do you interact with it in this course?', 'What problems with LLMs does RAG help solve, like missing knowledge or hallucinated answers?', "What are the main steps you'll build in the first part of the module to make the FAQ agent work?", 'How is the agentic version in part 2 different from the fixed pipeline built in part 1?'])

In [9]:
from evaluation_utils import llm_structured

In [10]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What is a Retrieval-Augmented Generation system, in simple terms?', 'Why does this course treat the language model like a black box instead of explaining how it works inside?', 'What are the main problems with using an LLM by itself for answering questions?', 'How does RAG help when the model doesn’t know the answer or can’t access your files?', 'What will be built in this module, and how is Part 1 different from Part 2?']


In [58]:
# Tracking cost
usage.input_tokens, usage.output_tokens

(1020, 109)

In [59]:
# try it for the second file
doc = documents[1]
doc

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

In [66]:
# Prepare the document as JSON
import json

doc = documents[1]

user_prompt = json.dumps(doc)

In [67]:
# Create the messages
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [68]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What do I need installed before I can follow this module, and is Python alone enough?', 'How do I create the new project from scratch with uv and install the packages the lesson uses?', 'What is the recommended way to keep my OpenAI key safe in this setup?', 'How do I start Jupyter and check that the OpenAI client is working in a notebook?', 'If I want to use Groq or another OpenAI-style API, what changes do I need to make to the key and client code?']


In [69]:
# Tracking cost
usage.input_tokens, usage.output_tokens

(1286, 116)

### The full ground truth

In [11]:
# load the generated questions for all the lessons
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [9]:
len(ground_truth)

360

##  Chunking and search by hand

In [12]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

In [26]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [27]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [14]:
# 1. Extract text content from your list of dictionaries
text_chunks = [chunk["content"] for chunk in chunks]

In [15]:
# 2. Embed all chunks (returns the 2D matrix directly)
vectors = embed.encode_batch(text_chunks)

In [16]:
vectors

array([[-0.08756469,  0.0183638 , -0.0812242 , ...,  0.03053823,
        -0.02172769,  0.032775  ],
       [ 0.02436195, -0.10619476,  0.03307316, ...,  0.01430084,
        -0.00125544,  0.04325692],
       [-0.01780485,  0.03103091,  0.00856107, ...,  0.02220214,
        -0.03375531,  0.04288228],
       ...,
       [ 0.0098034 ,  0.04912257,  0.01207492, ..., -0.09453997,
        -0.06321277,  0.04775798],
       [-0.03622024,  0.06821856, -0.01540893, ..., -0.00271634,
         0.01875559,  0.01007466],
       [-0.02975659, -0.00552576, -0.03531849, ...,  0.01044234,
         0.02297962, -0.01966067]], shape=(295, 384))

In [17]:
import numpy as np
X = np.array(vectors)

In [18]:
X.shape

(295, 384)

In [19]:
X

array([[-0.08756469,  0.0183638 , -0.0812242 , ...,  0.03053823,
        -0.02172769,  0.032775  ],
       [ 0.02436195, -0.10619476,  0.03307316, ...,  0.01430084,
        -0.00125544,  0.04325692],
       [-0.01780485,  0.03103091,  0.00856107, ...,  0.02220214,
        -0.03375531,  0.04288228],
       ...,
       [ 0.0098034 ,  0.04912257,  0.01207492, ..., -0.09453997,
        -0.06321277,  0.04775798],
       [-0.03622024,  0.06821856, -0.01540893, ..., -0.00271634,
         0.01875559,  0.01007466],
       [-0.02975659, -0.00552576, -0.03531849, ...,  0.01044234,
         0.02297962, -0.01966067]], shape=(295, 384))

In [17]:
# 3. Embed your single query 
query_text = "How does approximate nearest neighbor search work?"
q1_vector = embed.encode(query_text)

In [18]:
scores = X.dot(q1_vector)

In [19]:
scores

array([ 3.15187705e-01,  2.01479593e-01,  5.90559560e-02, -7.67661858e-02,
        1.18452480e-01, -1.41697042e-01, -2.81406552e-02, -4.65669225e-02,
       -2.06994704e-02, -6.07744087e-02,  2.13273853e-01,  8.87601799e-02,
       -1.97269351e-02,  3.11630016e-01,  5.51034674e-01,  2.04008048e-01,
        2.12515842e-01,  1.93649180e-01,  2.51961293e-01,  1.31078643e-01,
        2.59120579e-01,  7.63816008e-02,  9.59193707e-02,  9.81472975e-03,
       -3.59106882e-02,  1.01211577e-02,  1.10786937e-01, -9.90259208e-02,
       -3.71170151e-02,  7.59057570e-02, -3.35340540e-02,  8.86841309e-03,
        1.02636405e-01,  6.89614876e-02,  1.29408856e-01,  2.57709091e-01,
        3.23680614e-01,  1.06350075e-01,  5.61891367e-02,  2.34017457e-01,
        1.97954387e-01,  9.64296290e-02,  1.93709917e-01,  2.16719271e-01,
        3.48340456e-01,  5.10906092e-02,  2.05212837e-01,  1.05416170e-01,
       -3.25432514e-02,  4.94665548e-02,  2.38574865e-01, -3.44207108e-02,
        1.82165438e-01,  

In [37]:
import numpy as np

#Find the filename of the highest-scoring chunk
best_idx = np.argmax(scores)
winning_filename = chunks[best_idx]["filename"]

print(f"Highest Score: {scores[best_idx]:.4f}")
print(f"Belongs to file: {winning_filename}")

Highest Score: 0.6489
Belongs to file: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [38]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489016436447387))

In [39]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Vector search with minsearch

In [20]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [21]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [22]:
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Text search vs vector search

- We will try text search vs vector search for the same query



Index the same chunks with `Index` from minsearch. Use `content` as a text field.



In [21]:
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

- Text search results:

In [22]:
question = "How do I store vectors in PostgreSQL?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

- Vector search results:

In [23]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [24]:
# Take the first question from the ground truth:
q = ground_truth[0]["question"]

In [25]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [26]:
# First result with text search

search_results = index.search(
    q,
    num_results=5
)

search_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [30]:
# First result with vector search

query_vector = embed.encode(q)

results = vindex.search(query_vector, num_results=5)
results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [27]:
def text_search(query):

    return index.search(
        query,
        num_results=5
    )

In [32]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [28]:
def vector_search(query):
    query_vector = embed.encode(query)

    return vindex.search(query_vector, num_results=5)

In [34]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [32]:
doc = ground_truth[0]
doc

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [34]:
doc_id = doc["filename"]
results = text_search(query=doc["question"])
results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [36]:
# Compare the retrieved document IDs with the correct document ID
for d in results:
    print(f'{d["filename"]} == {doc_id}: {d["filename"] == doc_id}')

01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/01-intro.md == 01-agentic-rag/lessons/01-intro.md: True


In [37]:
# Then turn this comparison into a relevance list. 
# relevance means whether a retrieved document is the correct filename for this question

relevance = []

for d in results:
    relevance.append(int(d["filename"] == doc_id))

relevance


[0, 0, 0, 0, 1]

In [38]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [39]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 1]

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

In [41]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [42]:
# Call it for the first 15 ground truth questions
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [43]:
relevance_total_text

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [45]:
# make it generic for any search function, not just text search. For example, for vector search / hybrid search
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [47]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [48]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [49]:
# run it for all ground truth questions
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [50]:
relevance_total

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [51]:
# Hit Rate
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [52]:
hit_rate(relevance_total)

0.7583333333333333

In [53]:
# for vector search

relevance_total = compute_relevance_total(ground_truth_sample, vector_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [54]:
# run it for all ground truth questions [vector search]
relevance_total = compute_relevance_total(ground_truth, vector_search)
relevance_total

  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 1, 0],
 [0, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 1, 1, 1, 0],
 [1, 1, 0, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0,

In [55]:
# for vector search
hit_rate(relevance_total)

0.725

In [56]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [57]:
# for vector search
mrr(relevance_total)

0.5486111111111112

## Hybrid search

Reciprocal Rank Fusion (RRF)

In [58]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [59]:
# Text search results

question = "How do I give the model access to tools?"

text_results = index.search(
    question,
    num_results=5
)

text_results

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can

In [60]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
vector_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [61]:
results = rrf([vector_results, text_results])

In [62]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [ ]:
def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    results = rrf([vector_results, text_results])
    return results

In [65]:
# make it generic for any search function, not just text search. For example, for vector search / hybrid search
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [66]:
# compute_relevance
q = ground_truth[50]
print(q["question"])

Why do fixed RAG pipelines break down when the search step doesn’t find the right info?


In [67]:
compute_relevance(q, hybrid_search)

[1, 0, 0, 0, 0]

In [68]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [69]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total(ground_truth_sample, hybrid_search)

  0%|          | 0/15 [00:00<?, ?it/s]

In [70]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 0, 0, 1, 1],
 [1, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0]]

In [71]:
# total relevance for hybrid search

relevance_total_hybrid = compute_relevance_total(ground_truth, hybrid_search)
relevance_total_hybrid

  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 0, 0, 1, 1],
 [1, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 1],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0,

In [72]:
k = [1, 50, 100, 200]